In [1]:
%load_ext autoreload
%autoreload 2

import investos as inv
import numpy as np
import pandas as pd


## Generate Data


In [8]:
%load_ext autoreload
%autoreload 2

from humbldata.core.standard_models.openbbapi.EquityPriceHistoricalQueryParams import (
    EquityPriceHistoricalQueryParams,
)
from humbldata.core.utils.openbb_api_client import OpenBBAPIClient
from humbldata.toolbox.toolbox_helpers import log_returns


async def prepare_actual_returns_for_investos(
    symbols: str = "SPY",
    start_date: str = "2018-06-20", # XLC starts on 2018-06-20
    end_date: str = "2025-10-10",
    provider: str = "yfinance",
    adjustment: str = "splits_and_dividends",
) -> pd.DataFrame:
    api_query_params = EquityPriceHistoricalQueryParams(
        symbol=symbols,
        start_date=start_date,
        end_date=end_date,
        provider=provider,
        adjustment=adjustment,
    )
    api_client = OpenBBAPIClient()
    api_response = await api_client.fetch_data(
        obb_path="equity.price.historical",
        api_query_params=api_query_params,
    )
    equity_historical_data = api_response.to_polars(collect=False)
    log_returns_df = log_returns(equity_historical_data, _column_name="close")
    log_returns_pd = log_returns_df.collect().to_pandas()

    # Handle single symbol case - add symbol column if it doesn't exist
    if "symbol" not in log_returns_pd.columns:
        single_symbol = symbols.strip()
        log_returns_pd["symbol"] = single_symbol

    log_returns_pivot = log_returns_pd.pivot(index="date", columns="symbol", values="log_returns")
    log_returns_pivot_clean = log_returns_pivot.dropna().copy()
    log_returns_pivot_clean.loc[:, "cash"] = 0.0
    return log_returns_pivot_clean

# Example usage in notebook:
actual_returns = await prepare_actual_returns_for_investos()
actual_returns


INFO: OpenBBAPIClient || START: fetch_data (async)
INFO: OpenBBAPIClient || Prepared request for: https://data.humblfinance.io/api/v1/equity/price/historical?provider=yfinance&symbol=SPY&start_date=2018-06-20&end_date=2025-10-10&adjustment=splits_and_dividends
INFO: RateLimiter || Checking Rate Limits for Provider: yfinance Route: /equity/price/historical | 10/10 remaining (resets at 2025-10-13T11:10:23.000825)
INFO: RateLimiter || Updating Rate Limit - Provider: yfinance Route: /equity/price/historical | 9/10 remaining (resets at 2025-10-13T11:10:23.000825)
INFO: OpenBBAPIClient || Fetching data from: https://data.humblfinance.io/api/v1/equity/price/historical?provider=yfinance&symbol=SPY&start_date=2018-06-20&end_date=2025-10-10&adjustment=splits_and_dividends


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


INFO: OpenBBAPIClient || END: fetch_data (async) - Total time: 1.8901s


symbol,SPY,cash
date,,
2018-06-21,-0.006289,0.0
2018-06-22,0.001821,0.0
2018-06-25,-0.013706,0.0
2018-06-26,0.002212,0.0
2018-06-27,-0.008319,0.0
...,...,...
2025-10-06,0.003580,0.0
2025-10-07,-0.003714,0.0
2025-10-08,0.005945,0.0


In [9]:
%load_ext autoreload
%autoreload 2

from humbldata.toolbox.toolbox_controller import Toolbox

toolbox = Toolbox(
    symbols=["XLE", "XLF", "XLU", "XLI", "XLK", "XLV", "XLY", "XLP", "XLRE", "XLB", "XLC"],
    start_date="2018-06-20",
    end_date="2025-10-10",
    membership="admin",
)
humbl_compass = await toolbox.fundamental.humbl_compass(country="united_states", z_score="3m")
compass_metric = humbl_compass.to_pandas()
compass_metric

INFO: HumblCompassFetcher || START: fetch_data (async)
INFO: LogCacheHitPlugin || humbl_compass cache HIT & RETURNED [remote redis]
DEBUG: LogCacheHitPlugin || humbl_compass cache key: {"country": "united_states", "end_date": "2025-10-10", "start_date": "2018-06-20", "z_score": "3m"} [remote redis]
INFO: HumblCompassFetcher || END: fetch_data (async) - Total time: 0.0188s


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,date_month_start,country,cpi,cpi_3m_delta,cli,cli_3m_delta,humbl_regime,cpi_zscore,cli_zscore
0,2018-03-01,United States,2.36,0.25,100.89,0.33,humblBOUNCE,1.01,0.92
1,2018-04-01,United States,2.46,0.39,100.92,0.22,humblBOUNCE,0.94,0.80
2,2018-05-01,United States,2.80,0.59,100.90,0.08,humblBOUNCE,1.13,-0.27
3,2018-06-01,United States,2.87,0.51,100.85,-0.04,humblBLOAT,0.73,-1.12
4,2018-07-01,United States,2.95,0.49,100.78,-0.14,humblBLOAT,1.02,-1.07
...,...,...,...,...,...,...,...,...,...
85,2025-04-01,United States,2.31,-0.69,100.20,-0.11,humblBUST,-0.72,-0.99
86,2025-05-01,United States,2.35,-0.47,100.20,-0.10,humblBUST,0.07,-0.53
87,2025-06-01,United States,2.67,0.28,100.25,-0.00,humblBLOAT,1.15,1.15
88,2025-07-01,United States,2.70,0.39,100.30,0.09,humblBOUNCE,0.67,1.02


In [11]:
%load_ext autoreload
%autoreload 2

from humbldata.portfolio.strategies.humbl_compass_simple_backtest import (
    HumblCompassSimpleBacktest,
)

# Run HumblCompassSimpleBacktest with existing actual_returns and monthly compass metric
backtest = HumblCompassSimpleBacktest(
    actual_returns=actual_returns,
    compass_metric=compass_metric,
    regime_alignment="tradable_asof",  # options: "tradable_asof" or "month_attribution"
    lag_months=2,
    carry_past_last_known=False,
    cash_column_name="cash",
)

result = backtest.generate_hisotrical_performance_backtest()
result.summary

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,symbol,metric,bucket,num_periods,total_return,cumulative_return,annualized_return,annualized_excess_return,annualized_volatility,risk_over_cash_annualized,excess_risk_annualized,sharpe_ratio,information_ratio,max_drawdown,portfolio_hit_rate,annual_turnover
0,SPY,humblCOMPASS,humblBLOAT,356.0,-0.010348,-0.010348,-0.001469,-0.001469,0.189632,0.189632,0.189632,0.056143,0.056143,-0.272130,0.511236,None
1,SPY,humblCOMPASS,humblBOOM,290.0,0.127487,0.127487,0.030537,0.030537,0.130460,0.130460,0.130460,0.864581,0.864581,-0.101697,0.548276,None
2,SPY,humblCOMPASS,humblBOUNCE,582.0,0.228342,0.228342,0.028556,0.028556,0.250843,0.250843,0.250843,0.481290,0.481290,-0.357459,0.572165,None
3,SPY,humblCOMPASS,humblBUST,609.0,0.673763,0.673763,0.078319,0.078319,0.172293,0.172293,0.172293,1.323661,1.323661,-0.196726,0.561576,None
